# 05 · Previsões com o modelo vigente e comparação de ciclos

Carrega o modelo vigente pelo manifesto (o `Pipeline` já traz o pré-processamento e o conjunto de features) e
prevê **2025**, o teste reservado: registros que **nunca** participaram da seleção de features, do treino ou da
promoção. Depois compara o ciclo atual com os anteriores e demonstra o alerta do monitor de drift.

In [1]:
import sys, json, pathlib, warnings
RAIZ = pathlib.Path.cwd().resolve()
while not (RAIZ / "config" / "config.yaml").exists():  # funciona rodando de notebooks/ ou da raiz
    RAIZ = RAIZ.parent
sys.path.insert(0, str(RAIZ))
warnings.filterwarnings("ignore", message=".*Hint: Inferred schema.*")

import pandas as pd
from src.comum import carregar_config, caminho, estado, ler_ciclo, atualizar_ciclo, configurar_mlflow, tags_ciclo, pasta_evidencias, gatilho
cfg = carregar_config()
pd.set_option("display.width", 160)

In [2]:
from src import ingestao, modelagem as md, promocao as pr, preprocessamento as pp, drift
ciclo = ler_ciclo(cfg)
vigente, manifesto = pr.carregar_vigente(cfg)
v = manifesto["vigente"]
print(f"vigente: {v['algoritmo']} | ciclo {v['cycle_id']} | {v['data_version']} | features: {v['features']}")

vigente: hist_gradient_boosting | ciclo 20261124-0010-45294862870 | vbase-3f4e3604 | features: ['taxa_historica_periodo', 'acid_ult_365d', 'lon', 'delegacia', 'br', 'trecho_km', 'taxa_historica', 'lat', 'vmd', 'chuva_mm']


## Previsões para 2025 (teste reservado)

O histórico do trecho de cada dia de 2025 precisa dos dias anteriores, então a base é montada com todos os
acidentes até a véspera (base inicial + todos os lotes + 2025). Só as linhas de 2025 são avaliadas.

In [3]:
trechos = pd.read_parquet(caminho(cfg["dados"]["trechos"]))
clima = pd.read_parquet(caminho(cfg["dados"]["clima"]))
todos = ingestao.acidentes_limpos(cfg, ingestao.lotes_disponiveis(cfg), incluir_teste=True)
fim = todos["data"].max()
base = pp.montar(pp.tratar(todos, trechos, fim), trechos, clima)
teste = base[base["data"] >= pd.Timestamp(cfg["dados"]["inicio_teste"])].copy()
teste["prob"] = md.prever(vigente, teste, v["features"])
m_teste = md.metricas(teste[pp.ALVO].to_numpy(), teste["prob"].to_numpy())
print(f"{len(teste):,} linhas de 2025 ({teste[pp.ALVO].mean():.2%} com acidente)")
m_teste

  10 acidentes com coordenada fora de SC descartados


363,540 linhas de 2025 (2.10% com acidente)


{'roc_auc': 0.8182,
 'pr_auc': 0.1349,
 'brier': 0.01928,
 'razao_prevista_observada': 0.936,
 'captura_top_10pct': 0.5005,
 'captura_top_5pct': 0.3639}

In [4]:
nomes = trechos.set_index("trecho_id")
top = (teste.groupby(["trecho_id", "periodo"]).agg(prob_media=("prob", "mean"), periodos_com_acidente=(pp.ALVO, "sum"))
       .sort_values("prob_media", ascending=False).head(10).reset_index())
top["local"] = top["trecho_id"].map(lambda t: f"BR-{nomes.at[t, 'br']} km {nomes.at[t, 'trecho_km']} ({nomes.at[t, 'municipio']})")
top[["local", "periodo", "prob_media", "periodos_com_acidente"]]

,local,periodo,prob_media,periodos_com_acidente
0,BR-101 km 200 (SAO JOSE),manha,0.352221,150
1,BR-101 km 200 (SAO JOSE),tarde,0.351977,127
2,BR-101 km 210 (PALHOCA),tarde,0.258388,83
3,BR-101 km 210 (PALHOCA),noite,0.257826,88
4,BR-101 km 210 (PALHOCA),manha,0.257497,89
5,BR-101 km 200 (SAO JOSE),noite,0.224185,124
6,BR-101 km 130 (BALNEARIO CAMBORIU),tarde,0.180491,82
7,BR-101 km 130 (BALNEARIO CAMBORIU),noite,0.179975,69
8,BR-101 km 130 (BALNEARIO CAMBORIU),manha,0.179533,67
9,BR-101 km 140 (ITAPEMA),tarde,0.164646,71


## Previsão linha a linha

O trecho de maior risco no último dia de 2025, nos quatro períodos: as features brutas entram no `Pipeline` e
sai a probabilidade.

In [5]:
exemplo = teste[(teste["trecho_id"] == top.at[0, "trecho_id"])].tail(4)[["trecho_id", "data", "periodo", *v["features"]]]
exemplo.assign(prob=md.prever(vigente, exemplo, v["features"]))

,trecho_id,data,periodo,taxa_historica_periodo,acid_ult_365d,lon,delegacia,br,trecho_km,taxa_historica,lat,vmd,chuva_mm,prob
173988,101-200,2025-12-31,madrugada,0.188071,574.0,-48.619,DEL01-SC,101,200,1.595253,-27.584752,48285.5678,32.0,0.166796
173989,101-200,2025-12-31,manha,0.482958,574.0,-48.619,DEL01-SC,101,200,1.595253,-27.584752,48285.5678,0.7,0.349957
173990,101-200,2025-12-31,tarde,0.489957,574.0,-48.619,DEL01-SC,101,200,1.595253,-27.584752,48285.5678,1.8,0.352838
173991,101-200,2025-12-31,noite,0.434267,574.0,-48.619,DEL01-SC,101,200,1.595253,-27.584752,48285.5678,1.8,0.224516


## Comparação entre ciclos

In [6]:
hist = pd.DataFrame(manifesto["historico"])
cols = ["cycle_id", "data_version", "gatilho", "lotes_novos", "vencedor_ciclo", "valor_vencedor", "valor_vigente", "decisao", "vigente_apos"]
hist[[c for c in cols if c in hist]]

,cycle_id,data_version,gatilho,lotes_novos,vencedor_ciclo,valor_vencedor,valor_vigente,decisao,vigente_apos
0,20261124-0010-45294862870,vbase-3f4e3604,workflow_dispatch,[],hist_gradient_boosting,0.1274,NaN,promovido,20261124-0010-45294862870
1,20261124-2008-45477532512,v2024-01-a37b9b7d,schedule,[2024-01],hist_gradient_boosting,0.1334,0.1325,mantido,20261124-0010-45294862870


In [7]:
if len(hist) >= 2:
    a, b = hist.iloc[-2], hist.iloc[-1]
    print("features que entraram:", sorted(set(b["features_eleitas"]) - set(a["features_eleitas"])) or "nenhuma")
    print("features que saíram:  ", sorted(set(a["features_eleitas"]) - set(b["features_eleitas"])) or "nenhuma")
    print("PR-AUC de validação dos candidatos:")
    print(pd.DataFrame({"anterior": {k: c["metricas"]["pr_auc"] for k, c in a["candidatos"].items()},
                        "atual": {k: c["metricas"]["pr_auc"] for k, c in b["candidatos"].items()}}))

features que entraram: ['nebulosidade', 'periodo_idx']
features que saíram:   ['chuva_mm', 'trecho_km']
PR-AUC de validação dos candidatos:
                        anterior   atual
logistica                 0.1196  0.1264
hist_gradient_boosting    0.1274  0.1334


In [8]:
mlflow = configurar_mlflow(cfg)
runs = mlflow.search_runs(experiment_names=[cfg["projeto"]["experimento_mlflow"]])
cols = ["tags.cycle_id", "tags.data_version", "tags.etapa", "tags.algoritmo", "tags.gatilho", "tags.decisao", "metrics.val_pr_auc"]
runs[[c for c in cols if c in runs]].head(20)

,tags.cycle_id,tags.data_version,tags.etapa,tags.algoritmo,tags.gatilho,tags.decisao,metrics.val_pr_auc
0,20261124-2008-45477532512,v2024-01-a37b9b7d,promocao,NaN,schedule,mantido,NaN
1,20261124-2008-45477532512,v2024-01-a37b9b7d,treino,hist_gradient_boosting,schedule,mantido,0.1334
2,20261124-2008-45477532512,v2024-01-a37b9b7d,treino,logistica,schedule,NaN,0.1264
3,20261124-2008-45477532512,v2024-01-a37b9b7d,votacao,NaN,schedule,NaN,NaN
4,20261124-2008-45477532512,v2024-01-a37b9b7d,etl,NaN,schedule,NaN,NaN
5,20261124-0010-45294862870,vbase-3f4e3604,previsao,NaN,workflow_dispatch,NaN,NaN
6,20261124-0010-45294862870,vbase-3f4e3604,promocao,NaN,workflow_dispatch,promovido,NaN
7,20261124-0010-45294862870,vbase-3f4e3604,treino,hist_gradient_boosting,workflow_dispatch,promovido,0.1274
8,20261124-0010-45294862870,vbase-3f4e3604,treino,logistica,workflow_dispatch,NaN,0.1196
9,20261124-0010-45294862870,vbase-3f4e3604,votacao,NaN,workflow_dispatch,NaN,NaN


## Demonstração do monitor de drift (critério 10)

O mesmo monitor do notebook 01, aplicado a **abril de 2020** (início da pandemia, com queda de tráfego) contra
abril dos anos anteriores, e a um mês normal de 2024. O primeiro deve disparar o alerta e o segundo não.

In [9]:
demo = {}
for mes in ["2020-04-01", "2024-04-01"]:
    ref, atual = drift.recorte_mensal(base, pd.Timestamp(mes))
    r = drift.monitorar(ref, atual, pp.CANDIDATAS, cfg)
    demo[mes[:7]] = {"alerta": r["alerta"], "psi_max": r["psi_max"], "razao_taxa": r["razao_taxa"], "motivos": "; ".join(r["motivos"])}
pd.DataFrame(demo).T

,alerta,psi_max,razao_taxa,motivos
2020-04,True,0.3587,0.709,taxa de acidentes 0.71× a do mesmo mês nos ano...
2024-04,False,0.4682,1.014,


In [10]:
ev = pasta_evidencias(cfg, ciclo)
teste[["trecho_id", "data", "periodo", pp.ALVO, "prob"]].sort_values("prob", ascending=False).head(200).to_csv(ev / "previsoes_2025_top200.csv", index=False)
(ev / "metricas_teste_2025.json").write_text(json.dumps(m_teste, indent=2))
with mlflow.start_run(run_name=f"{ciclo['cycle_id']} · previsões 2025", tags=tags_ciclo(ciclo, etapa="previsao", modelo_vigente=v["cycle_id"])):
    mlflow.log_params({"vigente_cycle_id": v["cycle_id"], "vigente_algoritmo": v["algoritmo"], "vigente_run_id": v["run_id"]})
    mlflow.log_metrics({f"teste_{k}": val for k, val in m_teste.items()})
    mlflow.log_artifact(ev / "previsoes_2025_top200.csv")